# Statistics practice: churn and categorical variables

**Question.** Which categorical customer attributes are associated with churn,
and how strong is each association?

**Method.** For each variable, a chi-square test of independence against
`Churned`, with Cramér's V as the effect size.

**Caveats.**
- Association is not causation. These tests show that two variables move
  together, not that one drives the other.
- Large samples make p-values small. With 7,043 customers, a weak association
  can be highly significant, so read V for strength and p only for whether the
  pattern is distinguishable from chance.
- Variables overlap. Contract, internet type and support services are linked,
  so each test alone can overstate what that variable contributes.

In [1]:
import numpy as np
from scipy import stats

In [4]:
import pandas as pd
df = pd.read_csv("churn_features.csv")

In [6]:

# n = 3875
n = (df["Contract"] == "Month-to-month").sum()
# churned = 1655
churned = ((df["Contract"] == "Month-to-month") & (df["Churned"] == 1)).sum()
print(n,churned)
p = churned / n # churn rate
se = np.sqrt(p* (1 - p)/ n) # standart error
low, high = p - 1.96 * se, p + 1.96 * se
print(f"{p:.3f}, 95% CI {low:.3f} to {high:.3f}")

3875 1655
0.427, 95% CI 0.412 to 0.443


Month-to-month customers churned at 42.7% (1,655 of 3,875). The 95% interval of 41.2% to 44.3% reflects sampling error only. It doesn't show that contract type causes churn.

In [5]:
ct = pd.crosstab(df["Contract"], df["Churned"])
chi2,p_value, dof, _ = stats.chi2_contingency(ct)
print(ct)
print(f"chi-square {chi2:.1f}, p_value {p_value:.3g}")

Churned            0     1
Contract                  
Month-to-month  2220  1655
One year        1307   166
Two year        1647    48
chi-square 1184.6, p_value 5.86e-258


Churn differs by contract type (chi-square 1184.6, df = 2, p < 0.001), so the pattern is very unlikely to be sampling noise.

In [14]:
# Cramér's V
n_total = ct.values.sum()
v = np.sqrt(chi2 / (n_total * (min(ct.shape) - 1)))
print(f"Cramer's V {v:.2f}")

Cramer's V 0.41


Churn differs by contract type (χ² = 1184.6, df = 2, p < 0.001), and the association is moderate to strong (Cramér's V = 0.41)

In [38]:
n=(df["InternetService"] == "Fiber optic").sum()

churned= ((df["InternetService"] == "Fiber optic")  & (df["Churned"] == 1)).sum()
print(n ,churned)
p = churned/ n
se = np.sqrt(p *(1-p)/n)
low, high = p - 1.96* se ,p + 1.96* se
print(f"Fiber optic: n={n}, churned= {churned}, rate={p:.3f}, 95% CI ,{low:.3f} to  {high:.3f}")



3096 1297
Fiber optic: n=3096, churned= 1297, rate=0.419, 95% CI ,0.402 to  0.436


In [31]:
ct = pd.crosstab(df["InternetService"],df["Churned"])
chi2,p_value,dof,_ = stats.chi2_contingency(ct)
print(ct)

print(f"chi-square:{chi2:.3f}, p_value:{p_value:.3g}")


Churned             0     1
InternetService            
DSL              1962   459
Fiber optic      1799  1297
No               1413   113
chi-square:732.310, p_value:9.57e-160


In [ ]:
# Crammer's V
n_total = ct.values.sum()
v = np.sqrt(chi2 / ( n_total * (min(ct.shape) - 1)))
print(f"Crammer's {v:.2f}")

Crammer's 0.32


Churn differs by internet service (χ² = 732.3, df = 2, p < 0.001), a moderate association (Cramér's V = 0.32). Contract type is the stronger association (V = 0.41), and most of the fibre gap reflects contract mix.

In [ ]:
#  a universal function

def association(df, column, target="Churned"):
    """Chi-square test asn Cramer's V for one categorical columns against the target."""
    ct = pd.crosstab(df[column],df[target])
    chi2, p_value,dof, _ = stats.chi2_contingency(ct)
    n_total = ct.values.sum()
    v = np.sqrt(chi2 / (n_total * (min(ct.shape)-1)))
    print(f"{column}: chi-square {chi2:.1f}, df {dof}, p {p_value:.3g}, Cramer's V {v:.2f}")
    return {"column" : column, "chi2":chi2, "p_value": p_value,"dof" : dof,"cramers_v":v}


In [46]:
results = []
for col in ["Contract", "InternetService","PaymentMethod","TechSupport","OnlineSecurity"]:
    results.append(association(df,col))

summary = pd.DataFrame(results).sort_values("cramers_v",ascending=False)
summary = summary[["column","chi2", "dof", "p_value", "cramers_v"]]
summary.round({"chi2": 1, "p_value": 3, "cramers_v": 2})

Contract: chi-square 1184.6, df 2, p 5.86e-258, Cramer's V 0.41
InternetService: chi-square 732.3, df 2, p 9.57e-160, Cramer's V 0.32
PaymentMethod: chi-square 648.1, df 3, p 3.68e-140, Cramer's V 0.30
TechSupport: chi-square 828.2, df 2, p 1.44e-180, Cramer's V 0.34
OnlineSecurity: chi-square 850.0, df 2, p 2.66e-185, Cramer's V 0.35


,column,chi2,dof,p_value,cramers_v
0,Contract,1184.6,2,0.0,0.41
4,OnlineSecurity,850.0,2,0.0,0.35
3,TechSupport,828.2,2,0.0,0.34
1,InternetService,732.3,2,0.0,0.32
2,PaymentMethod,648.1,3,0.0,0.30


In [43]:
internet = df[df["InternetService"] != "No"]
association(internet, "OnlineSecurity")
association(internet, "TechSupport")

OnlineSecurity: chi-square 433.8, df 1, p 2.41e-96, Cramer's V 0.28
TechSupport: chi-square 414.3, df 1, p 4.35e-92, Cramer's V 0.27


{'column': 'TechSupport',
 'chi2': np.float64(414.2515690453345),
 'p_value': np.float64(4.35191667899426e-92),
 'dof': 1,
 'cramers_v': np.float64(0.2740189444521598)}

Online security and tech support are both associated with churn among customers who have internet (V = 0.28 and 0.27). Part of their overall association comes from the "no internet" group, which churns less, but the association remains after restricting to internet customers.

In [45]:
for contract in ["Month-to-month", "One year", "Two year"]:
    subset = internet[internet["Contract"] == contract]
    print(f"---- {contract} (n={len(subset)}) ----")
    association(subset, "OnlineSecurity")
    association(subset, "TechSupport")

---- Month-to-month (n=3351) ----
OnlineSecurity: chi-square 103.8, df 1, p 2.21e-24, Cramer's V 0.18
TechSupport: chi-square 82.7, df 1, p 9.5e-20, Cramer's V 0.16
---- One year (n=1109) ----
OnlineSecurity: chi-square 9.2, df 1, p 0.00237, Cramer's V 0.09
TechSupport: chi-square 0.2, df 1, p 0.648, Cramer's V 0.01
---- Two year (n=1057) ----
OnlineSecurity: chi-square 7.3, df 1, p 0.00698, Cramer's V 0.08
TechSupport: chi-square 2.1, df 1, p 0.145, Cramer's V 0.04


Among customers with internet, support services are associated with churn mainly among month-to-month customers (Cramér's V 0.16 to 0.18). Within one- and two-year contracts the association is weak or absent, so much of the support effect reflects contract type.